# initializing pyspark and javapath

In [1]:
from pyspark.sql import SparkSession
import os

# Set JAVA_HOME to a Homebrew-installed Java runtime
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk21/Contents/Home'
os.environ['PATH'] = os.path.join(os.environ['JAVA_HOME'], 'bin') + ':' + os.environ.get('PATH', '')



In [2]:
spark = SparkSession.builder \
    .appName("MyApp") \
    .config("spark.dynamicAllocation.enabled", "false") \
    .getOrCreate()

spark

26/06/22 23:24:52 WARN Utils: Your hostname, Apples-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.4 instead (on interface en0)
26/06/22 23:24:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/22 23:24:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Data import

In [3]:
df = spark.read.option("header", "true").csv("/Users/apple/Desktop/worldcup-dbt/worldcup_dbt/seeds/matches.csv")


1. Data understanding
            Start by answering basic questions about the dataset.

            Questions
            How many rows are in the dataset?

            What columns exist?

            Which columns have nulls?

            Which columns should be numeric but are stored as strings?

            Are there duplicate rows?

In [4]:
print(df.explain(extended=True))

== Parsed Logical Plan ==
Relation [match_id#17,year#18,home_team#19,away_team#20,home_goals#21,away_goals#22,stage#23,country#24,attendance#25] csv

== Analyzed Logical Plan ==
match_id: string, year: string, home_team: string, away_team: string, home_goals: string, away_goals: string, stage: string, country: string, attendance: string
Relation [match_id#17,year#18,home_team#19,away_team#20,home_goals#21,away_goals#22,stage#23,country#24,attendance#25] csv

== Optimized Logical Plan ==
Relation [match_id#17,year#18,home_team#19,away_team#20,home_goals#21,away_goals#22,stage#23,country#24,attendance#25] csv

== Physical Plan ==
FileScan csv [match_id#17,year#18,home_team#19,away_team#20,home_goals#21,away_goals#22,stage#23,country#24,attendance#25] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/apple/Desktop/worldcup-dbt/worldcup_dbt/seeds/matches.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<match_id:string,year:strin

In [5]:
from pyspark.sql.functions import col, when, count, avg, min, max, countDistinct, collect_list, collect_set, round, abs, sum, desc, asc

understanding the data

In [6]:
"""

            How many rows are in the dataset?

            What columns exist?

            Which columns have nulls?

            Which columns should be numeric but are stored as strings?
            
            """

df.printSchema()
df.describe().show()
df.select([
    count(when(col(c).isNull(), 1)).alias(c) for c in df.columns
]).show()

root
 |-- match_id: string (nullable = true)
 |-- year: string (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- home_goals: string (nullable = true)
 |-- away_goals: string (nullable = true)
 |-- stage: string (nullable = true)
 |-- country: string (nullable = true)
 |-- attendance: string (nullable = true)



26/06/22 23:24:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+-----------------+---------+---------+------------------+------------------+-----------+-----------+-----------------+
|summary|          match_id|             year|home_team|away_team|        home_goals|        away_goals|      stage|    country|       attendance|
+-------+------------------+-----------------+---------+---------+------------------+------------------+-----------+-----------+-----------------+
|  count|               100|              100|      100|      100|               100|               100|        100|        100|              100|
|   mean|              50.5|          2012.62|     NULL|     NULL|              1.64|               0.8|       NULL|       NULL|         57088.89|
| stddev|29.011491975882016|6.768584591606862|     NULL|     NULL|1.1592404340077103|0.8989331499509894|       NULL|       NULL|17828.86300577485|
|    min|                 1|             2002|Argentina|  Algeria|                 0|                 0|      Final|  

In [7]:
print("Row : ", df.count(),"\nColumn :",len(df.columns))
df.columns

Row :  100 
Column : 9


['match_id',
 'year',
 'home_team',
 'away_team',
 'home_goals',
 'away_goals',
 'stage',
 'country',
 'attendance']

data cleaning

In [8]:
df1 = df.na.drop(how = 'any')


In [9]:
df1.select([
    count(when(col(c).isNull(), 1)).alias(c) for c in df.columns
]).show()

+--------+----+---------+---------+----------+----------+-----+-------+----------+
|match_id|year|home_team|away_team|home_goals|away_goals|stage|country|attendance|
+--------+----+---------+---------+----------+----------+-----+-------+----------+
|       0|   0|        0|        0|         0|         0|    0|      0|         0|
+--------+----+---------+---------+----------+----------+-----+-------+----------+



In [10]:
df1.tail(3)

[Row(match_id='98', year='2019', home_team='USA', away_team='England', home_goals='2', away_goals='1', stage='Semi Final', country='France', attendance='53512'),
 Row(match_id='99', year='2019', home_team='France', away_team='USA', home_goals='2', away_goals='1', stage='Quarter Final', country='France', attendance='25302'),
 Row(match_id='100', year='2019', home_team='Netherlands', away_team='Italy', home_goals='2', away_goals='0', stage='Quarter Final', country='France', attendance='21669')]

In [11]:
print("no duplicates") if df1.dropDuplicates().count()/df1.count() == 1.0 else print("duplicates exist")

no duplicates


In [12]:
df2 = df1.withColumn('match_id', col('match_id').cast('int')) \
         .withColumn('attendance', col('attendance').cast('int')) \
         .withColumn('home_goals', col('home_goals').cast('int') ) \
         .withColumn('away_goals', col('away_goals').cast('int'))

df2.filter(col('match_id').isNull() | col('attendance').isNull()).show()
df2.printSchema()

#if casting fails the dataset will be having string values, identify them and resolve it or seperate them
#mostly remove the null records or fill them using fillna, imputer, or other methods that is suitable

+--------+----+---------+---------+----------+----------+-----+-------+----------+
|match_id|year|home_team|away_team|home_goals|away_goals|stage|country|attendance|
+--------+----+---------+---------+----------+----------+-----+-------+----------+
+--------+----+---------+---------+----------+----------+-----+-------+----------+

root
 |-- match_id: integer (nullable = true)
 |-- year: string (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- home_goals: integer (nullable = true)
 |-- away_goals: integer (nullable = true)
 |-- stage: string (nullable = true)
 |-- country: string (nullable = true)
 |-- attendance: integer (nullable = true)




- Show all matches from 2022.


In [13]:
games_2022 = df2.filter(col('year') >= 2022)
games_2022.show()

+--------+----+-----------+-----------+----------+----------+-------------+-------+----------+
|match_id|year|  home_team|  away_team|home_goals|away_goals|        stage|country|attendance|
+--------+----+-----------+-----------+----------+----------+-------------+-------+----------+
|      79|2022|  Argentina|     France|         3|         3|        Final|  Qatar|     88966|
|      80|2022|    Croatia|    Morocco|         2|         1|  Third Place|  Qatar|     44137|
|      81|2022|  Argentina|    Croatia|         3|         0|   Semi Final|  Qatar|     88966|
|      82|2022|     France|    Morocco|         2|         0|   Semi Final|  Qatar|     44667|
|      83|2022|Netherlands|  Argentina|         2|         2|Quarter Final|  Qatar|     88235|
|      84|2022|    England|     France|         1|         2|Quarter Final|  Qatar|     88235|
|      85|2022|    Croatia|     Brazil|         1|         1|Quarter Final|  Qatar|     88103|
|      86|2022|    Morocco|   Portugal|         1|


- Show all matches where Brazil played.


In [14]:
brazil_games = df2.filter((col('home_team') == 'Brazil') | (col('away_team') == 'Brazil'))
brazil_games.show()

+--------+----+-----------+-----------+----------+----------+-------------+------------+----------+
|match_id|year|  home_team|  away_team|home_goals|away_goals|        stage|     country|attendance|
+--------+----+-----------+-----------+----------+----------+-------------+------------+----------+
|       6|2018|     Brazil|    Belgium|         1|         2|Quarter Final|      Russia|     42792|
|      13|2018|     Brazil|     Mexico|         2|         0|  Round of 16|      Russia|     42877|
|      18|2014|Netherlands|     Brazil|         3|         0|  Third Place|      Brazil|     68034|
|      19|2014|    Germany|     Brazil|         7|         1|   Semi Final|      Brazil|     58141|
|      22|2014|     Brazil|   Colombia|         2|         1|Quarter Final|      Brazil|     68551|
|      25|2014|     Brazil|      Chile|         1|         1|  Round of 16|      Brazil|     57714|
|      37|2010|Netherlands|     Brazil|         2|         1|Quarter Final|South Africa|     53026|


-- brazil has a 3:2 win ratio across the worldcup fiesta


- Show matches where total goals were more than 5.



In [15]:
fiveplus_goals = df2.filter(((col('home_goals')+col('away_goals')) > 5))
fiveplus_goals.show()

+--------+----+---------+---------+----------+----------+-----------+-------+----------+
|match_id|year|home_team|away_team|home_goals|away_goals|      stage|country|attendance|
+--------+----+---------+---------+----------+----------+-----------+-------+----------+
|       1|2018|   France|  Croatia|         4|         2|      Final| Russia|     78011|
|       9|2018|   France|Argentina|         4|         3|Round of 16| Russia|     44052|
|      19|2014|  Germany|   Brazil|         7|         1| Semi Final| Brazil|     58141|
|      79|2022|Argentina|   France|         3|         3|      Final|  Qatar|     88966|
+--------+----+---------+---------+----------+----------+-----------+-------+----------+



-- france is a goal scoring machine

- Show matches with attendance above 80,000.

In [16]:
top_attendance = df2.filter(col('attendance') > 79999)
top_attendance.show()

+--------+----+-----------+-----------+----------+----------+-------------+------------+----------+
|match_id|year|  home_team|  away_team|home_goals|away_goals|        stage|     country|attendance|
+--------+----+-----------+-----------+----------+----------+-------------+------------+----------+
|      33|2010|      Spain|Netherlands|         1|         0|        Final|South Africa|     84490|
|      39|2010|  Argentina|    Germany|         0|         4|Quarter Final|South Africa|     83391|
|      43|2010|  Argentina|     Mexico|         3|         1|  Round of 16|South Africa|     84377|
|      79|2022|  Argentina|     France|         3|         3|        Final|       Qatar|     88966|
|      81|2022|  Argentina|    Croatia|         3|         0|   Semi Final|       Qatar|     88966|
|      83|2022|Netherlands|  Argentina|         2|         2|Quarter Final|       Qatar|     88235|
|      84|2022|    England|     France|         1|         2|Quarter Final|       Qatar|     88235|



- Which team scored the most goals as home team?



In [17]:
df2.groupBy('home_team','year').agg(max('home_goals').alias('top_scorer')).orderBy(col("top_scorer").desc()).show(1)

+---------+----+----------+
|home_team|year|top_scorer|
+---------+----+----------+
|  Germany|2014|         7|
+---------+----+----------+
only showing top 1 row



- Which team scored the most goals as away team?



In [18]:
df2.groupby('away_team','year').agg(max('away_goals').alias('awy_topscr')).orderBy(col("awy_topscr").desc()).show(1)

+---------+----+----------+
|away_team|year|awy_topscr|
+---------+----+----------+
|  Germany|2010|         4|
+---------+----+----------+
only showing top 1 row



- What is the average attendance by year?



In [19]:
df2.groupBy('year').max('attendance').show()

+----+---------------+
|year|max(attendance)|
+----+---------------+
|2019|          57900|
|2014|          74738|
|2002|          69029|
|2018|          78011|
|2006|          72000|
|2022|          88966|
|2010|          84490|
+----+---------------+



- Which year had the highest average goals per match?


In [20]:
df2.withColumn("total_goals", col("home_goals") + col("away_goals")) \
    .groupBy("year") \
    .agg(round(avg("total_goals"), 2).alias("total_goals_match")) \
    .orderBy(col("total_goals_match").desc()) \
    .show(1)

+----+-----------------+
|year|total_goals_match|
+----+-----------------+
|2022|              3.0|
+----+-----------------+
only showing top 1 row




- Which stage had the highest attendance?

In [21]:
df2.groupby("stage").agg(max("attendance").alias("max_attendance"), min("attendance").alias("min_attendance"),round(avg("attendance"),2).alias("average_fans")).show()

+-------------+--------------+--------------+------------+
|        stage|max_attendance|min_attendance|average_fans|
+-------------+--------------+--------------+------------+
|  Third Place|         68034|         20316|    49262.43|
|        Final|         88966|         57900|    74590.57|
|  Round of 16|         88103|         25176|    52238.82|
|  Group Stage|         41056|         41056|     41056.0|
|Quarter Final|         88966|         21669|    59980.23|
|   Semi Final|         88966|         44667|    63616.36|
+-------------+--------------+--------------+------------+



In [22]:
df2.printSchema()

root
 |-- match_id: integer (nullable = true)
 |-- year: string (nullable = true)
 |-- home_team: string (nullable = true)
 |-- away_team: string (nullable = true)
 |-- home_goals: integer (nullable = true)
 |-- away_goals: integer (nullable = true)
 |-- stage: string (nullable = true)
 |-- country: string (nullable = true)
 |-- attendance: integer (nullable = true)




- Who won each match?


In [23]:
df2.show(5)

+--------+----+---------+---------+----------+----------+-------------+-------+----------+
|match_id|year|home_team|away_team|home_goals|away_goals|        stage|country|attendance|
+--------+----+---------+---------+----------+----------+-------------+-------+----------+
|       1|2018|   France|  Croatia|         4|         2|        Final| Russia|     78011|
|       2|2018|  Belgium|  England|         2|         0|  Third Place| Russia|     60613|
|       3|2018|   France|  Belgium|         1|         0|   Semi Final| Russia|     74894|
|       4|2018|  Croatia|  England|         2|         1|   Semi Final| Russia|     78011|
|       5|2018|  Uruguay|   France|         0|         2|Quarter Final| Russia|     42873|
+--------+----+---------+---------+----------+----------+-------------+-------+----------+
only showing top 5 rows



In [24]:
df3 = df2.withColumn("winner", when(col("home_goals") > col("away_goals"), col("home_team")).otherwise(col("away_team"))) 
df3. select(col("home_team"), col("away_team"), col("winner")) \
    .show()

+-----------+-----------+-----------+
|  home_team|  away_team|     winner|
+-----------+-----------+-----------+
|     France|    Croatia|     France|
|    Belgium|    England|    Belgium|
|     France|    Belgium|     France|
|    Croatia|    England|    Croatia|
|    Uruguay|     France|     France|
|     Brazil|    Belgium|    Belgium|
|     Sweden|    England|    England|
|     Russia|    Croatia|    Croatia|
|     France|  Argentina|     France|
|    Uruguay|   Portugal|    Uruguay|
|     Russia|      Spain|      Spain|
|    Croatia|    Denmark|    Denmark|
|     Brazil|     Mexico|     Brazil|
|    Belgium|      Japan|    Belgium|
|     Sweden|Switzerland|     Sweden|
|    England|   Colombia|   Colombia|
|    Germany|  Argentina|    Germany|
|Netherlands|     Brazil|Netherlands|
|    Germany|     Brazil|    Germany|
|  Argentina|Netherlands|Netherlands|
+-----------+-----------+-----------+
only showing top 20 rows




- Was the match a home win, away win, or draw?


In [25]:
df4 = df3.withColumn("win_side", when(col("away_goals") == col("home_goals"), "Draw") \
               .when(col("home_goals") > col("away_goals"), "Home Win") \
                .otherwise("Away Win")) 

df4.select(["home_team", "away_team","home_goals", "away_goals" ,"winner", "win_side"]).show()

+-----------+-----------+----------+----------+-----------+--------+
|  home_team|  away_team|home_goals|away_goals|     winner|win_side|
+-----------+-----------+----------+----------+-----------+--------+
|     France|    Croatia|         4|         2|     France|Home Win|
|    Belgium|    England|         2|         0|    Belgium|Home Win|
|     France|    Belgium|         1|         0|     France|Home Win|
|    Croatia|    England|         2|         1|    Croatia|Home Win|
|    Uruguay|     France|         0|         2|     France|Away Win|
|     Brazil|    Belgium|         1|         2|    Belgium|Away Win|
|     Sweden|    England|         0|         2|    England|Away Win|
|     Russia|    Croatia|         2|         2|    Croatia|    Draw|
|     France|  Argentina|         4|         3|     France|Home Win|
|    Uruguay|   Portugal|         2|         1|    Uruguay|Home Win|
|     Russia|      Spain|         1|         1|      Spain|    Draw|
|    Croatia|    Denmark|         


- What is the goal difference?


In [26]:
df4 = df4.withColumn("goal_dif", abs(col("home_goals")-col("away_goals")))
df4.select(["home_team", "away_team", "goal_dif"]).show()

+-----------+-----------+--------+
|  home_team|  away_team|goal_dif|
+-----------+-----------+--------+
|     France|    Croatia|       2|
|    Belgium|    England|       2|
|     France|    Belgium|       1|
|    Croatia|    England|       1|
|    Uruguay|     France|       2|
|     Brazil|    Belgium|       1|
|     Sweden|    England|       2|
|     Russia|    Croatia|       0|
|     France|  Argentina|       1|
|    Uruguay|   Portugal|       1|
|     Russia|      Spain|       0|
|    Croatia|    Denmark|       0|
|     Brazil|     Mexico|       2|
|    Belgium|      Japan|       1|
|     Sweden|Switzerland|       1|
|    England|   Colombia|       0|
|    Germany|  Argentina|       1|
|Netherlands|     Brazil|       3|
|    Germany|     Brazil|       6|
|  Argentina|Netherlands|       0|
+-----------+-----------+--------+
only showing top 20 rows




- What is the total goals scored?

In [27]:
df4 = df4.withColumn("goal_total", abs(col("home_goals")+col("away_goals")))
df4.select(["home_team", "away_team", "goal_total"]).show()

+-----------+-----------+----------+
|  home_team|  away_team|goal_total|
+-----------+-----------+----------+
|     France|    Croatia|         6|
|    Belgium|    England|         2|
|     France|    Belgium|         1|
|    Croatia|    England|         3|
|    Uruguay|     France|         2|
|     Brazil|    Belgium|         3|
|     Sweden|    England|         2|
|     Russia|    Croatia|         4|
|     France|  Argentina|         7|
|    Uruguay|   Portugal|         3|
|     Russia|      Spain|         2|
|    Croatia|    Denmark|         2|
|     Brazil|     Mexico|         2|
|    Belgium|      Japan|         5|
|     Sweden|Switzerland|         1|
|    England|   Colombia|         2|
|    Germany|  Argentina|         1|
|Netherlands|     Brazil|         3|
|    Germany|     Brazil|         8|
|  Argentina|Netherlands|         0|
+-----------+-----------+----------+
only showing top 20 rows




- Top 10 matches by attendance



In [28]:

df4.orderBy(col("attendance").desc()).show(10)

+--------+----+-----------+-----------+----------+----------+-------------+-------+----------+-----------+--------+--------+----------+
|match_id|year|  home_team|  away_team|home_goals|away_goals|        stage|country|attendance|     winner|win_side|goal_dif|goal_total|
+--------+----+-----------+-----------+----------+----------+-------------+-------+----------+-----------+--------+--------+----------+
|      86|2022|    Morocco|   Portugal|         1|         0|Quarter Final|  Qatar|     88966|    Morocco|Home Win|       1|         1|
|      79|2022|  Argentina|     France|         3|         3|        Final|  Qatar|     88966|     France|    Draw|       0|         6|
|      81|2022|  Argentina|    Croatia|         3|         0|   Semi Final|  Qatar|     88966|  Argentina|Home Win|       3|         3|
|      83|2022|Netherlands|  Argentina|         2|         2|Quarter Final|  Qatar|     88235|  Argentina|    Draw|       0|         4|
|      84|2022|    England|     France|         

- Top 10 highest-scoring matches


In [29]:
df4.withColumn("total_goals", (col("home_goals")+col("away_goals"))).orderBy(col("total_goals").desc()).show(10)

+--------+----+---------+-----------+----------+----------+-----------+------------+----------+-----------+--------+--------+----------+-----------+
|match_id|year|home_team|  away_team|home_goals|away_goals|      stage|     country|attendance|     winner|win_side|goal_dif|goal_total|total_goals|
+--------+----+---------+-----------+----------+----------+-----------+------------+----------+-----------+--------+--------+----------+-----------+
|      19|2014|  Germany|     Brazil|         7|         1| Semi Final|      Brazil|     58141|    Germany|Home Win|       6|         8|          8|
|       9|2018|   France|  Argentina|         4|         3|Round of 16|      Russia|     44052|     France|Home Win|       1|         7|          7|
|       1|2018|   France|    Croatia|         4|         2|      Final|      Russia|     78011|     France|Home Win|       2|         6|          6|
|      79|2022|Argentina|     France|         3|         3|      Final|       Qatar|     88966|     France

In [30]:
#df4.filter(col("win_side") == "Draw").show() 


- Top 10 teams by total goals scored


In [31]:
df_home = df4.select(col("home_team").alias("team"), col("home_goals").alias("goals"))
df_away = df4.select(col("away_team").alias("team"), col("away_goals").alias("goals"))

df4_teamgl = df_home.union(df_away)
#df4_teamgl.withColumn("goals", col("goals").cast('int'))
df4_teamgl.groupBy("team").agg(sum("goals").alias("total_goals")) \
    .orderBy(desc("total_goals")).show(10)

+-----------+-----------+
|       team|total_goals|
+-----------+-----------+
|     France|         33|
|    Germany|         31|
|     Brazil|         24|
|  Argentina|         23|
|Netherlands|         20|
|    England|         16|
|    Croatia|         11|
|    Belgium|          9|
|    Uruguay|          9|
|        USA|          8|
+-----------+-----------+
only showing top 10 rows




- Which team appears most often as home team?



In [32]:
(df4.groupBy("home_team").count()).orderBy(desc("count")).show(8)

+-----------+-----+
|  home_team|count|
+-----------+-----+
|     France|   12|
|    Germany|   11|
|  Argentina|    9|
|Netherlands|    9|
|     Brazil|    9|
|      Spain|    6|
|    England|    6|
|    Uruguay|    5|
+-----------+-----+
only showing top 8 rows




- Which team appears most often overall?


In [33]:
(df4_teamgl.groupby("team").count()).orderBy(desc("count")).show(5)

+-----------+-----+
|       team|count|
+-----------+-----+
|    Germany|   16|
|     France|   16|
|     Brazil|   15|
|  Argentina|   14|
|Netherlands|   13|
+-----------+-----+
only showing top 5 rows



In [34]:
df4_teamgl.columns

['team', 'goals']


- Which team won the most matches?


In [35]:
(df4.groupBy("winner").count()).orderBy(desc("count")).show(5)

+-----------+-----+
|     winner|count|
+-----------+-----+
|     France|   15|
|    Germany|   12|
|  Argentina|    9|
|     Brazil|    9|
|Netherlands|    9|
+-----------+-----+
only showing top 5 rows




- Which stage has the most matches?


In [36]:
(df4.groupBy("stage").count()).orderBy(desc("count")).show(5)

+-------------+-----+
|        stage|count|
+-------------+-----+
|  Round of 16|   45|
|Quarter Final|   26|
|   Semi Final|   14|
|  Third Place|    7|
|        Final|    7|
+-------------+-----+
only showing top 5 rows




- Which country hosted the most matches?

In [37]:
(df4.groupBy("country").count()).orderBy(desc("count")).show()

+------------+-----+
|     country|count|
+------------+-----+
|      Russia|   16|
|     Germany|   16|
|       Qatar|   16|
|      Brazil|   16|
|South Africa|   16|
| South Korea|   14|
|      France|    6|
+------------+-----+




- How many home wins, away wins, and draws are there?


In [38]:
df_win_pct = df4.groupby( "win_side" ).count()
df_win_pct.show()

+--------+-----+
|win_side|count|
+--------+-----+
|    Draw|   21|
|Away Win|   13|
|Home Win|   66|
+--------+-----+




- What is the win percentage for home teams?


In [39]:
sum1 = df_win_pct.agg(sum(col("count"))).collect()[0][0]
df_win_pct = df_win_pct.withColumn("pct", col("count")*100/sum1)
df_win_pct.filter(col("win_side") == "Home Win").show()


+--------+-----+----+
|win_side|count| pct|
+--------+-----+----+
|Home Win|   66|66.0|
+--------+-----+----+




- Which team has the highest win count?

In [40]:
df4.groupBy(col("winner")).agg((count("winner")).alias("win_count")).orderBy(col("win_count").desc()).show(1)

+------+---------+
|winner|win_count|
+------+---------+
|France|       15|
+------+---------+
only showing top 1 row




- For each year, what was the average attendance?


In [41]:
df1.groupby("year").agg(avg("attendance")).show()

+----+-----------------+
|year|  avg(attendance)|
+----+-----------------+
|2019|          37858.5|
|2014|         56544.25|
|2002|47616.71428571428|
|2018|       53058.3125|
|2006|          64875.0|
|2022|       71915.1875|
|2010|         54551.25|
+----+-----------------+




- For each country, what was the average number of goals per match?


In [42]:
df4.groupby("country").agg(round(avg((col("home_goals")+col("away_goals"))),2).alias("goals_per_game")).show()

+------------+--------------+
|     country|goals_per_game|
+------------+--------------+
|      Russia|          2.94|
|     Germany|          1.88|
|      France|          2.33|
|       Qatar|           3.0|
| South Korea|          1.86|
|      Brazil|          2.19|
|South Africa|          2.75|
+------------+--------------+




- Which teams consistently appear in high-attendance matches?


In [43]:
df4.describe("attendance").show()

+-------+-----------------+
|summary|       attendance|
+-------+-----------------+
|  count|              100|
|   mean|         57088.89|
| stddev|17828.86300577485|
|    min|            20316|
|    max|            88966|
+-------+-----------------+




- Which stage has the highest average goals?


In [44]:
from pyspark.sql.functions import array, explode

(df4.filter(col("attendance") > 60000)
    .select(explode(array(col("home_team"), col("away_team"))).alias("team"))) \
        .groupby("team").count().orderBy(desc("count")).show(3)

+---------+-----+
|     team|count|
+---------+-----+
|Argentina|   11|
|   France|    9|
|  Germany|    8|
+---------+-----+
only showing top 3 rows



In [45]:
df4.groupBy("stage").agg(round(avg((col("home_goals")+col("away_goals"))),2).alias("goals_stage")).orderBy(desc("goals_stage")).show()

+-------------+-----------+
|        stage|goals_stage|
+-------------+-----------+
|  Third Place|       3.57|
|        Final|       2.86|
|  Round of 16|       2.53|
|   Semi Final|       2.29|
|Quarter Final|        2.0|
|  Group Stage|        1.0|
+-------------+-----------+




- What is the correlation between attendance and total goals?

In [46]:
# What is the correlation between attendance and total goals?

df4.withColumn("goals", (col("home_goals")+col("away_goals")).cast("int")) \
    .select(["goals","attendance"]).groupby("goals") \
        .agg(round(avg("attendance"),2).alias("attendance")) \
            .orderBy("goals").show()

+-----+----------+
|goals|attendance|
+-----+----------+
|    0|  56477.14|
|    1|  58954.35|
|    2|  54092.42|
|    3|  53562.11|
|    4|   69632.5|
|    5|  55568.67|
|    6|   83488.5|
|    7|   44052.0|
|    8|   58141.0|
+-----+----------+



## Final analysis

- The dataset shows a clear increase in average attendance in the most recent World Cups, with 2022 having the highest average attendance in the notebook output.
- Overall attendance is strong but variable: the sample mean is about 57k, with values ranging from roughly 20k to nearly 89k.
- Higher-scoring matches do not always attract more spectators. Some 4- and 6-goal games had very high attendance, but the 7-goal games were lower, so the relationship between goals and attendance is not linear.
- By stage, the Third Place match and Final had the highest average goals per game, while the Group Stage had the lowest, which suggests knockout matches tend to be more open or more decisive.
- Teams like Argentina, France, and Germany appeared most often in the high-attendance matches, which suggests marquee teams help draw larger crowds.

Overall, the data suggests that attendance is influenced more by tournament stage, host context, and popular teams than by total goals alone.
